In [1]:
%pip install numpy mne osl-dynamics tensorflow ipyevents ipywidgets pyvista pyvistaqt pandas seaborn nibabel nilearn scikit-image numba pyyaml scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\marce\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import sys
!conda run -n hmmnet pip install h5io

'conda' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
import mne
import numpy as np
import scipy.io
import h5py

from osl_dynamics import analysis, inference, files
from osl_dynamics.meeg import preproc, rhino, source_recon, parcellation
from osl_dynamics.data import Data
from osl_dynamics.utils import plotting
from osl_dynamics.utils.filenames import OSLFilenames


In [ ]:
import h5py
import numpy as np
import mne
import matplotlib.pyplot as plt
from scipy.signal import welch

# =============================================================================
# 1. LOAD DATA FROM MAT FILE
# =============================================================================

subject_folder = '//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data/s05/'
subject_filename = 'Subject5_spm_mne'

filepath = subject_folder + subject_filename + '.mat'

with h5py.File(filepath, 'r') as f:
    fid_pnt    = np.array(f['fid']['fid']['pnt']).T
    hsp_pnt    = np.array(f['fid']['pnt']).T
    data       = np.array(f['data']).T        # (n_channels, n_times)
    sfreq      = float(np.array(f['fs']).flatten()[0])
    chanpos    = np.array(f['sens']['chanpos']).T
    chanori    = np.array(f['sens']['chanori']).T
    chan_names = []
    for ref in f['chan_names'][:, 0]:
        name = ''.join(chr(c) for c in f[ref][:].flatten())
        chan_names.append(name)
    print(f['sens'])



print(f"Data shape: {data.shape}")
print(f"N channels: {len(chan_names)}")
print(f"Sampling rate: {sfreq} Hz")
print(f"EEG channel count: {len(eeg_names)}")
print(f"EEG data shape: {data_eeg.shape}")
print(eeg_names)

<HDF5 group "/sens" (13 members)>
Data shape: (341, 14801, 240)
N channels: 341
Sampling rate: 1000.0 Hz
EEG channel count: 4
EEG data shape: (4, 14801, 240)
['EEG057', 'EEG058', 'EEG059', 'EEG060']


In [ ]:
# =============================================================================
# 2. BUILD MNE RAW OBJECT
# =============================================================================
# Load CTF file for correct channel info (positions, coil types, dev_head_t)
raw_ctf = mne.io.read_raw_ctf(
    '//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data/s05/mg4670_katja2_20160623_01.ds',
    preload=False
)

# Strip CTF serial number suffix
ctf_ch_names_stripped = [ch.split('-')[0] for ch in raw_ctf.ch_names]

# Reorder SPM data to match CTF channel order and convert fT → T
spm_to_ctf_idx = [chan_names.index(ch) for ch in ctf_ch_names_stripped]
data_meg = data[spm_to_ctf_idx, :] / 1e15   # (273, n_times) in Tesla


# isolate EEG data from total channels
eeg_idx = [i for i, name in enumerate(chan_names) if name.startswith("EEG")]
eeg_names = [chan_names[i] for i in eeg_idx]
data_eeg = data[eeg_idx, :]

print(f"EEG data shape: {data_eeg.shape}")
print(f"EEG channel count: {len(eeg_names)}")
print(f"EEG data shape: {data_eeg.shape}")

# reorder data_eeg to have shape (n_epochs/trials, n_channels, n_times)#
data_eeg = data_eeg.transpose(1, 0, 2)
print("Reordered data_eeg shape:", data_eeg.shape)
print(f"EEG channel count: {len(eeg_names)}")
print(f"EEG data shape: {data_eeg.shape}")

########## Reorder chanpos to match CTF channel order
########## chanpos_meg = chanpos[spm_to_ctf_idx, :]
print(f"data_meg shape: {data_meg.shape}")
print(f"data_meg std: {data_meg.std():.2e}")  # should be ~1e-13



######### Create Raw using CTF info (has correct positions, coil types, dev_head_t)
######### raw = mne.io.RawArray(data_meg, raw_ctf.info)

# create MNE info object for EEG data
info = mne.create_info(
    ch_names=eeg_names,
    sfreq=sfreq,
    ch_types=["eeg"] * len(eeg_names),
)

# create MNE Epoch Array for EEG data (with shape (n_epochs, n_channels, n_times))
Epochs_data_array = mne.EpochsArray(data_eeg, info)
print(f"dev_head_t: {Epochs_data_array.info['dev_head_t']}")

ds directory : \\rdp.arc.ucl.ac.uk\ritd-ag-project-rd01pz-dbush99\Katja Button Press Data\s05\mg4670_katja2_20160623_01.ds
    res4 data read.
    hc data read.
    Separate EEG position data file read.
    Quaternion matching (desired vs. transformed):
      -0.02   76.26    0.00 mm <->   -0.02   76.26   -0.00 mm (orig :  -47.05   64.14 -266.25 mm) diff =    0.000 mm
       0.02  -76.26    0.00 mm <->    0.02  -76.26    0.00 mm (orig :   55.90  -47.62 -279.35 mm) diff =    0.000 mm
      99.56    0.00    0.00 mm <->   99.56   -0.00   -0.00 mm (orig :   77.83   74.42 -260.68 mm) diff =    0.000 mm
    Coordinate transformations established.
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
    4 EEG electrode locations assigned to channel info.
    4 EEG locations added to Polhemus data.
    Measurement info composed.
Finding samples for \\rdp.arc.ucl.ac.uk\ritd-ag-project-rd01pz-dbush99\Katja Button Press Data\s05\mg4670_katja2_20160623_01.d

In [24]:
# =============================================================================
# 3. SAVE
# =============================================================================
import os

os.chdir(subject_folder)        
outfile = subject_folder + subject_filename + '.fif'
Epochs_data_array.save(outfile, overwrite=True)
print("Saved!")


C:\Users\chick\AppData\Local\Temp\ipykernel_33404\3061489649.py:8: RuntimeWarning: This filename (//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data/s05/Subject5_spm_mne.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  Epochs_data_array.save(outfile, overwrite=True)


Saved!
